In [ ]:
import sys, os
from pathlib import Path

PROJECT_ROOT = Path(os.getcwd()).resolve()
while not (PROJECT_ROOT / 'setup.py').exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError('Cannot find project root')
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
import sys, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from solve_dae.integrate import solve_dae
import json

ROOT = Path("../..").resolve()
sys.path.insert(0, str(ROOT / "scripts" / "Experimental"))

from simulate_HFR import build_atol, soc_cutoff_event
from simulate_sensitivity import build_cell_op
from rfbmodel1D.parameters.params import membraneparams, cellparams, anodeparams, cathodeparams

: 

Parametric Sweep

In [ ]:
# -- User inputs ---------------------------------------------------------------
TEST_NO_SENS  = 16       # test case to load baseline from (1–29)
J_SENS        = 1500.0   # absolute current density [A/m²]  (charge = +J, discharge = -J)
DELTA_SENS    = 0.20     # fractional perturbation (0.20 = ±20 %)
TF_SENS       = 1800     # simulation duration per run [s]
RH_BASE_SENS  = 1.25     # baseline RH — not stored in HFR.csv
RTOL          = 1e-8

# Extract baseline operating point from HFR test data
# -- load experimental data and pick the point ---------------------------------
csv  = ROOT / "data" / "experimental" / "HFR.csv"
data = pd.read_csv(csv)
data = data[data["test_no"] == TEST_NO_SENS]

soc = data["SOC"].iloc[0]
T   = data["T(°C)"].iloc[0]
Q_v = data["Q_v (ml/min)"].iloc[0]

# -- fit results ----------------------------------------------------------------
# Path relative to scripts/Experimental/fit_results/, or None to use defaults
FIT_DIR = ROOT / "results" / "test fitting" / f"test {TEST_NO_SENS}"
HFR_FIT    = sorted(FIT_DIR.glob(f"fit_v2_test{TEST_NO_SENS}_*.json"))[-1].name
# -- apply HFR membrane params --------------------------------------------------
if HFR_FIT is not None:
    with open(FIT_DIR / HFR_FIT) as f:
        _hfr = json.load(f)
    _p = _hfr["params"]
    _s = _hfr["settings"]
    membraneparams.k_sorp_vw   = _p["k_vw_sorp"]
    membraneparams.k_desorp_vw = _p["S_desorp"] * _p["k_vw_sorp"]
    membraneparams.k_sorp_lw   = _p["k_lw_sorp"]
    membraneparams.k_desorp_lw = _p["S_desorp"] * _p["k_lw_sorp"]
    membraneparams.k_ca_lw     = _p["k_ca"]
    membraneparams.sf_D        = _s["sf_D"]
    cellparams.vanadium_uptake = False
    print(f"HFR membrane params loaded from: {HFR_FIT}")

# -- apply E-cell kinetic params ------------------------------------------------
fit_dir_cell  = ROOT / "results" / "test fitting" / "global" 
E_CELL_FIT    = sorted(fit_dir_cell.glob(f"global_k_ca_{int(T)}.json"))[-1].name
if E_CELL_FIT is not None:
    with open(fit_dir_cell / E_CELL_FIT) as f:
        _ec = json.load(f)
    k_ca  = _ec["params"]["k_ca"]
    j0_an = _ec["params"]["j0_an"][f"{TEST_NO_SENS}"]
    print(f"E-cell params loaded from: {E_CELL_FIT}")
    print("k_ca =", k_ca, "j0_an =", j0_an)
cathodeparams.reaction_rate_constant = k_ca
anodeparams.j_0                      = j0_an


In [ ]:
# -- Load baseline operating point from experimental data ----------------------
hfr_df        = pd.read_csv(ROOT / "data/experimental/HFR.csv")
_row          = hfr_df[hfr_df["test_no"] == TEST_NO_SENS].iloc[0]
SOC_BASE_SENS = float(_row["SOC"])
T_BASE_SENS   = float(_row["T(°C)"])
Q_BASE_SENS   = float(_row["Q_v (ml/min)"])

print(f"Baseline — Test {TEST_NO_SENS}: SOC={SOC_BASE_SENS:.2f}, "
      f"T={T_BASE_SENS:.1f} °C, Q_v={Q_BASE_SENS:.1f} ml/min, "
      f"RH={RH_BASE_SENS:.1f},  j=±{J_SENS:.0f} A/m²")


# -- Helper: extrapolate E_cell to the current-collector face -----------------
def _E_cell_from_sol(cell, sol, j_appl):
    Nc     = cell.mesh.cathode_mesh.n_x
    n_tank = cell.layout.n_vars_tank
    n_ca   = cell.layout.n_vars_cathode
    y_ca   = sol.y[n_tank:n_tank + Nc * n_ca, :].reshape(Nc, n_ca, -1)
    Phi_el = y_ca[:, 4, -1]
    dx0    = cell.mesh.cathode_mesh.dx[0] * cell.cathode.thickness
    return float(Phi_el[0] + j_appl * (dx0 / 2) / cell.cathode.eff_conductivity)


# -- Helper: run charge + discharge for one operating point --------------------
def run_point_sens(soc, T, Q_v, RH, j_appl):
    E = {}
    for j_signed, tag in [(+j_appl, "charge"), (-j_appl, "discharge")]:
        cell, y0, yp0 = build_cell_op(soc=soc, j_appl=j_signed, T=T, Q_v=Q_v, RH=RH)
        atol = build_atol(cell)
        sol  = solve_dae(
            cell.dae_residual_full_cell,
            (0.0, TF_SENS), y0, yp0,
            method="BDF", atol=atol, rtol=RTOL,
            events=soc_cutoff_event,
            jac_sparsity=cell.compute_jac_sparsity(y0, yp0),
        )
        E[tag] = _E_cell_from_sol(cell, sol, j_signed)
    VE = E["discharge"] / E["charge"]
    P  = j_appl * E["discharge"]        # W/m²
    return {**E, "VE": VE, "P": P}


# -- Parameter definitions -----------------------------------------------------
_base_kw = dict(soc=SOC_BASE_SENS, T=T_BASE_SENS, Q_v=Q_BASE_SENS,
                RH=RH_BASE_SENS, j_appl=J_SENS)

param_defs_sens = {
    "j_appl": {"label": f"j_appl  ({J_SENS:g} A/m²)",      "base": J_SENS,        "clip": None},
    "Q_v":    {"label": f"Q_v  ({Q_BASE_SENS:g} ml/min)",  "base": Q_BASE_SENS,   "clip": None},
    "T":      {"label": f"T  ({T_BASE_SENS:g} °C)",        "base": T_BASE_SENS,   "clip": None},
    "RH":     {"label": f"RH  ({RH_BASE_SENS:g})",         "base": RH_BASE_SENS,  "clip": None},
    "soc":    {"label": f"SOC  ({SOC_BASE_SENS:.2f})",     "base": SOC_BASE_SENS, "clip": (0.05, 0.95)},
}

# -- Run baseline --------------------------------------------------------------
print("\nRunning baseline ...")
_base_res = run_point_sens(**_base_kw)
print(f"  E_ch={_base_res['charge']:.4f} V   E_di={_base_res['discharge']:.4f} V   "
      f"VE={_base_res['VE']:.4f}   P={_base_res['P']:.2f} W/m²")

# -- Run perturbed cases -------------------------------------------------------
_sens_data = {}
for pkey, pcfg in param_defs_sens.items():
    lo_val = pcfg["base"] * (1 - DELTA_SENS)
    hi_val = pcfg["base"] * (1 + DELTA_SENS)
    if pcfg["clip"]:
        lo_val = float(np.clip(lo_val, *pcfg["clip"]))
        hi_val = float(np.clip(hi_val, *pcfg["clip"]))
    _sens_data[pkey] = {}
    for tag, val in [("lo", lo_val), ("hi", hi_val)]:
        pct = (val / pcfg["base"] - 1) * 100
        print(f"\n  {pkey} {tag}  = {val:.4g}  ({pct:+.0f}%) ...")
        res = run_point_sens(**{**_base_kw, pkey: val})
        _sens_data[pkey][tag] = res
        print(f"    E_ch={res['charge']:.4f} V   E_di={res['discharge']:.4f} V   "
              f"VE={res['VE']:.4f}   P={res['P']:.2f} W/m²")

print("\nDone.")

In [ ]:
from matplotlib.patches import Patch

# Exclude j_appl from the plot
EXCLUDE = {"j_appl"}

# Variable symbols for y-axis (LaTeX)
LABELS_SYM = {
    "Q_v": r"$Q_v$",
    "T":   r"$T$",
    "RH":  r"RH",
    "soc": r"SOC",
}

# Formatter for the bar end-labels
def _fmt(pkey, val):
    if pkey == "Q_v":
        return f"{val:.0f} mL min$^{{-1}}$"
    if pkey == "T":
        return f"{val:.1f} \u00b0C"
    if pkey == "RH":
        return f"{val:.2f}"
    if pkey == "soc":
        return f"{val:.2f}"
    return f"{val:.3g}"

params  = [p for p in param_defs_sens.keys() if p not in EXCLUDE]
labels  = [LABELS_SYM.get(p, p) for p in params]
base_VE = _base_res["VE"]
base_P  = _base_res["P"]

# Perturbed values (clipped where applicable)
pert_vals = np.array([
    [param_defs_sens[p]["base"] * (1 - DELTA_SENS),
     param_defs_sens[p]["base"] * (1 + DELTA_SENS)]
    for p in params
])
for i, p in enumerate(params):
    clip = param_defs_sens[p].get("clip")
    if clip:
        pert_vals[i] = np.clip(pert_vals[i], *clip)

# Relative-perturbation sensitivities
pVE = np.array([[(_sens_data[p]["lo"]["VE"] - base_VE) / (base_VE * DELTA_SENS),
                 (_sens_data[p]["hi"]["VE"] - base_VE) / (base_VE * DELTA_SENS)]
                for p in params])
pP  = np.array([[(_sens_data[p]["lo"]["P"]  - base_P)  / (base_P  * DELTA_SENS),
                 (_sens_data[p]["hi"]["P"]  - base_P)  / (base_P  * DELTA_SENS)]
                for p in params])

sort_idx  = np.argsort(np.abs(pVE[:, 1] - pVE[:, 0]))[::-1]
labels_s  = [labels[i] for i in sort_idx]
params_s  = [params[i] for i in sort_idx]
pVE_s     = pVE[sort_idx]
pP_s      = pP[sort_idx]
pert_s    = pert_vals[sort_idx]

COL_HI    = "#2A5783"
COL_LO    = "#A03B2D"
COL_LABEL = "0.45"          # soft grey for the perturbed-value annotations
H         = 0.36
X_LIM     = 0.15            # fixed x-axis limit for both panels
y_pos     = np.arange(len(params))

fig, axes = plt.subplots(1, 2, figsize=(11, 5.5))

# Map each panel to its metric identifier and x-axis label
panel_info = [
    (axes[0], pVE_s, r"$p_\mathrm{rel}$ — Voltage efficiency  [-]"),
    (axes[1], pP_s,  r"$p_\mathrm{rel}$ — Discharge power density  [-]"),
]

for ax, data_s, xlabel in panel_info:
    pad = 0.06 * X_LIM
    ax.set_xlim(-X_LIM - 5 * pad, X_LIM + 5 * pad)

    for i in range(len(params_s)):
        d_lo, d_hi = data_s[i]
        v_lo, v_hi = pert_s[i]

        ax.barh(i + H / 2, d_hi, height=H, color=COL_HI,
                edgecolor="black", linewidth=0.4, zorder=3, clip_on=False)
        ax.barh(i - H / 2, d_lo, height=H, color=COL_LO,
                edgecolor="black", linewidth=0.4, zorder=3, clip_on=False)

        ax.text(d_hi + (pad if d_hi >= 0 else -pad), i + H / 2,
                _fmt(params_s[i], v_hi),
                ha="left" if d_hi >= 0 else "right", va="center",
                fontsize=10, color=COL_LABEL, zorder=4)
        ax.text(d_lo + (pad if d_lo >= 0 else -pad), i - H / 2,
                _fmt(params_s[i], v_lo),
                ha="left" if d_lo >= 0 else "right", va="center",
                fontsize=10, color=COL_LABEL, zorder=4)

    ax.axvline(0, color="black", lw=0.8, zorder=5)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(labels_s, fontsize=13)
    ax.tick_params(axis="x", labelsize=11)
    ax.set_xlabel(xlabel, fontsize=12)
    ax.grid(True, axis="x", alpha=0.3, linewidth=0.5, zorder=0)

    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)

    ax.set_box_aspect(1)

handles = [
    Patch(facecolor=COL_HI, edgecolor="black", linewidth=0.4,
          label=f"+{DELTA_SENS*100:.0f}%"),
    Patch(facecolor=COL_LO, edgecolor="black", linewidth=0.4,
          label=f"\u2212{DELTA_SENS*100:.0f}%"),
]
fig.legend(handles=handles, fontsize=12, loc="lower center",
           ncol=2, frameon=False, bbox_to_anchor=(0.5, -0.02))

axes[0].text(-0.18, 1.02, "(a)", transform=axes[0].transAxes,
             fontsize=18, fontweight="bold", va="bottom")
axes[1].text(-0.18, 1.02, "(b)", transform=axes[1].transAxes,
             fontsize=18, fontweight="bold", va="bottom")

fig.tight_layout(rect=[0, 0.04, 1, 1])
plt.savefig(ROOT / "figures" / "sensitivity_analysis_op.pdf", bbox_inches="tight")
plt.show()

## Discharge power-density sensitivity: 3 tests x 3 current densities (9 panels)

Same one-at-a-time perturbation as the tornado chart above (`param_defs_sens`),
repeated at 3 x 3 = 9 discharge operating points instead of the single fixed
one: tests 5 / 16 / 25 (SOC low / mid / high), each at three discharge current
densities — two shared points (-500, -1500 A/m²) plus that test's own
**experimental maximum discharge current** (from `HFR.csv`).

Only the operating variables are perturbed (Q_v, T, RH, SOC) — `j_appl` is
what changes between panels, so it isn't also perturbed within a panel. Uses
a tighter `DELTA_SENS_J9 = 5%` (vs. `DELTA_SENS = 20%` above) so the
high-|j| points — closest to each test's mass-transport-limited knee — stay
inside the local-derivative regime, and the already-tight `RTOL` (1e-8, set
in the *Parameters* cell) avoids BDF step-error noise at those stiff points.

Each test supplies its own SOC / T / Q_v, HFR-fit membrane parameters, and
global-fit E-cell kinetics (`k_ca` shared, `j0_an` per test) — same loading
pattern as the single-condition baseline above, just repeated per test. RH
stays fixed at `RH_BASE_SENS` for all nine, matching the baseline case.

Only discharge power density `P = |j| x E_cell` is plotted (single metric,
not the VE + P pair above, since there is no charge leg here).

In [ ]:
def load_test_condition(test_no):
    """SOC / T / Q_v, HFR-fit membrane params, and global-fit E-cell kinetics for one test."""
    d = hfr_df[hfr_df["test_no"] == test_no].iloc[0]
    T_t = float(d["T(°C)"])

    fit_dir = ROOT / "results" / "test fitting" / f"test {test_no}"
    hfr_fit_path = sorted(fit_dir.glob(f"fit_v2_test{test_no}_*.json"))[-1]
    with open(hfr_fit_path) as f:
        _hfr = json.load(f)
    _p, _s = _hfr["params"], _hfr["settings"]

    fit_dir_cell = ROOT / "results" / "test fitting" / "global"
    ec_fit_path = sorted(fit_dir_cell.glob(f"global_k_ca_{int(T_t)}.json"))[-1]
    with open(ec_fit_path) as f:
        _ec = json.load(f)

    return dict(
        soc=float(d["SOC"]), T=T_t, Q_v=float(d["Q_v (ml/min)"]),
        hfr_fit_name=hfr_fit_path.name, ec_fit_name=ec_fit_path.name,
        k_sorp_vw=_p["k_vw_sorp"], k_desorp_vw=_p["S_desorp"] * _p["k_vw_sorp"],
        k_sorp_lw=_p["k_lw_sorp"], k_desorp_lw=_p["S_desorp"] * _p["k_lw_sorp"],
        k_ca_lw=_p["k_ca"], sf_D=_s["sf_D"],
        k_ca=_ec["params"]["k_ca"], j0_an=_ec["params"]["j0_an"][f"{test_no}"],
    )


def apply_test_condition(c):
    membraneparams.k_sorp_vw   = c["k_sorp_vw"]
    membraneparams.k_desorp_vw = c["k_desorp_vw"]
    membraneparams.k_sorp_lw   = c["k_sorp_lw"]
    membraneparams.k_desorp_lw = c["k_desorp_lw"]
    membraneparams.k_ca_lw     = c["k_ca_lw"]
    membraneparams.sf_D        = c["sf_D"]
    cellparams.vanadium_uptake = False
    cathodeparams.reaction_rate_constant = c["k_ca"]
    anodeparams.j_0                      = c["j0_an"]


def run_point_discharge(soc, T, Q_v, RH, j_appl):
    """Discharge-only counterpart of run_point_sens(): E_cell and power density."""
    cell, y0, yp0 = build_cell_op(soc=soc, j_appl=j_appl, T=T, Q_v=Q_v, RH=RH)
    atol = build_atol(cell)
    sol = solve_dae(
        cell.dae_residual_full_cell,
        (0.0, TF_SENS), y0, yp0,
        method="BDF", atol=atol, rtol=RTOL,
        events=soc_cutoff_event,
        jac_sparsity=cell.compute_jac_sparsity(y0, yp0),
    )
    E_di = _E_cell_from_sol(cell, sol, j_appl)
    return {"E_cell": E_di, "P": abs(j_appl) * E_di, "success": sol.success, "t_end": sol.t[-1]}

In [ ]:
# -- three tests x three discharge current densities (9 conditions) -----------
TESTS_OP_J = [
    {"label": "Low",  "test_no": 5},
    {"label": "Mid",  "test_no": 16},
    {"label": "High", "test_no": 25},
]
J_LABELS = ["-500 A/m2", "-1500 A/m2", "max discharge"]

for t in TESTS_OP_J:
    d = hfr_df[(hfr_df["test_no"] == t["test_no"]) & (hfr_df["mode"] == "discharge")]
    j_max = float(d["j_appl (A/m2)"].max())
    t["j_list"] = [-500.0, -1500.0, -j_max]
    print(f"Test {t['test_no']}: max discharge j_appl = {j_max:.1f} A/m2  ->  j_list = {t['j_list']}")

DELTA_SENS_J9 = 0.05   # tighter than DELTA_SENS (0.20) -- see markdown above

_results_9x9 = {}
for t in TESTS_OP_J:
    test_no, label = t["test_no"], t["label"]
    c = load_test_condition(test_no)
    apply_test_condition(c)
    print(f"\n{'='*70}\n{label}  |  test {test_no}\n{'='*70}")
    print(f"  SOC={c['soc']:.2f}  T={c['T']:.0f} degC  Q_v={c['Q_v']:.0f} ml/min  "
          f"HFR fit: {c['hfr_fit_name']}  E-cell fit: {c['ec_fit_name']}")

    param_defs_9 = {
        "Q_v": {"base": c["Q_v"],      "clip": None},
        "T":   {"base": c["T"],        "clip": None},
        "RH":  {"base": RH_BASE_SENS,  "clip": None},
        "soc": {"base": c["soc"],      "clip": (0.05, 0.95)},
    }

    for j, j_label in zip(t["j_list"], J_LABELS):
        base_kw = dict(soc=c["soc"], T=c["T"], Q_v=c["Q_v"], RH=RH_BASE_SENS, j_appl=j)
        print(f"\n  -- {j_label}  (j={j:+.1f} A/m2) --")
        base_res = run_point_discharge(**base_kw)
        print(f"    baseline: E_di={base_res['E_cell']:.4f} V   P={base_res['P']:.2f} W/m2")

        sens = {}
        for pkey, pcfg in param_defs_9.items():
            lo_val = pcfg["base"] * (1 - DELTA_SENS_J9)
            hi_val = pcfg["base"] * (1 + DELTA_SENS_J9)
            if pcfg["clip"]:
                lo_val = float(np.clip(lo_val, *pcfg["clip"]))
                hi_val = float(np.clip(hi_val, *pcfg["clip"]))
            sens[pkey] = {}
            for tag, val in [("lo", lo_val), ("hi", hi_val)]:
                res = run_point_discharge(**{**base_kw, pkey: val})
                sens[pkey][tag] = res
                print(f"    {pkey:5s} {tag}={val:.4g}: E_di={res['E_cell']:.4f} V  P={res['P']:.2f} W/m2")

        _results_9x9[(label, j_label)] = {
            "test_no": test_no, "j": j, "loaded": c,
            "base": base_res, "sens": sens, "param_defs": param_defs_9,
        }

print("\nAll 9 conditions done.")

In [ ]:
def _p_rel(entry, metric):
    param_defs = entry["param_defs"]
    ks = list(param_defs.keys())
    base = entry["base"][metric]
    return ks, np.array([
        [(entry["sens"][p]["lo"][metric] - base) / (base * DELTA_SENS_J9),
         (entry["sens"][p]["hi"][metric] - base) / (base * DELTA_SENS_J9)]
        for p in ks
    ])

# Shared parameter order, fixed from the High/max-discharge condition
_ref = _results_9x9[("High", J_LABELS[-1])]
_ref_keys, _pP_ref = _p_rel(_ref, "P")
sort_idx_9 = np.argsort(np.abs(_pP_ref[:, 1] - _pP_ref[:, 0]))[::-1]
params_9 = [_ref_keys[i] for i in sort_idx_9]
labels_9 = [LABELS_SYM.get(p, p) for p in params_9]

# Shared x-axis limit across all 9 panels
_all_p = np.concatenate([
    _p_rel(_results_9x9[(t["label"], jl)], "P")[1].ravel()
    for t in TESTS_OP_J for jl in J_LABELS
])
X_LIM_9 = 1.1 * np.max(np.abs(_all_p))

fig, axes = plt.subplots(3, 3, figsize=(15, 15))

for row, t in enumerate(TESTS_OP_J):
    for col, j_label in enumerate(J_LABELS):
        ax = axes[row, col]
        entry = _results_9x9[(t["label"], j_label)]
        keys, pP_full = _p_rel(entry, "P")
        idx_map = {k: i for i, k in enumerate(keys)}
        pP = pP_full[[idx_map[p] for p in params_9]]

        pert_vals_9 = np.array([
            (entry["param_defs"][p]["base"] * (1 - DELTA_SENS_J9),
             entry["param_defs"][p]["base"] * (1 + DELTA_SENS_J9))
            for p in params_9
        ])
        for i, p in enumerate(params_9):
            clip = entry["param_defs"][p].get("clip")
            if clip:
                pert_vals_9[i] = np.clip(pert_vals_9[i], *clip)

        y_pos = np.arange(len(params_9))
        pad = 0.06 * X_LIM_9
        ax.set_xlim(-X_LIM_9 - 5 * pad, X_LIM_9 + 5 * pad)
        for i in range(len(params_9)):
            d_lo, d_hi = pP[i]
            v_lo, v_hi = pert_vals_9[i]
            ax.barh(i + H / 2, d_hi, height=H, color=COL_HI,
                    edgecolor="black", linewidth=0.4, zorder=3, clip_on=False)
            ax.barh(i - H / 2, d_lo, height=H, color=COL_LO,
                    edgecolor="black", linewidth=0.4, zorder=3, clip_on=False)
            ax.text(d_hi + (pad if d_hi >= 0 else -pad), i + H / 2, _fmt(params_9[i], v_hi),
                    ha="left" if d_hi >= 0 else "right", va="center", fontsize=8, color=COL_LABEL, zorder=4)
            ax.text(d_lo + (pad if d_lo >= 0 else -pad), i - H / 2, _fmt(params_9[i], v_lo),
                    ha="left" if d_lo >= 0 else "right", va="center", fontsize=8, color=COL_LABEL, zorder=4)

        ax.axvline(0, color="black", lw=0.8, zorder=5)
        ax.set_yticks(y_pos)
        ax.set_yticklabels(labels_9 if col == 0 else [], fontsize=11)
        ax.tick_params(axis="x", labelsize=9)
        if row == 2:
            ax.set_xlabel(r"$p_\mathrm{rel}$ — Discharge power density  [-]", fontsize=10)
        ax.grid(True, axis="x", alpha=0.3, linewidth=0.5, zorder=0)
        for spine in ("top", "right"):
            ax.spines[spine].set_visible(False)
        ax.set_box_aspect(1)

        j_val = entry["j"]
        ax.set_title(f"{t['label']} (test {t['test_no']}) | {j_label}\n"
                     f"j={j_val:+.0f} A/m2   P={entry['base']['P']:.0f} W/m2",
                     fontsize=10)

handles = [
    Patch(facecolor=COL_HI, edgecolor="black", linewidth=0.4, label=f"+{DELTA_SENS_J9*100:.0f}%"),
    Patch(facecolor=COL_LO, edgecolor="black", linewidth=0.4, label=f"−{DELTA_SENS_J9*100:.0f}%"),
]
fig.legend(handles=handles, fontsize=12, loc="lower center", ncol=2, frameon=False,
           bbox_to_anchor=(0.5, -0.01))
fig.suptitle("Discharge power-density sensitivity across 3 tests x 3 current densities",
             fontsize=14, fontweight="bold", y=1.01)
fig.tight_layout(rect=[0, 0.02, 1, 1])
plt.savefig(ROOT / "figures" / "sensitivity_analysis_op_9x9.pdf", bbox_inches="tight")
plt.show()